# Chapter 3 - Finite Automata

## Learning Objectives

After completing this chapter, you will be able to:

* Define a finite automaton.
* Represent a finite automaton with a state diagram or a transition table.
* Work with and understand a finite automaton class built in Python.
* Define a language with a finite automaton.

We use three libraries to build and visualize the directed graphs we'll be using in this chapter:

| Package | Purpose |
|---|---|
| `io` | Basic library for handling input and output |
| `base64` | Manages special characters |
| `IPython` | Provides additional Python functionality |
| `graphviz` | General graph visualization |

The cell below checks whether each package is already installed and installs
it automatically if not. This is safe to re-run; it only installs what is
missing.

**Outside Jupyter**, install once from the terminal:
```bash
pip install io base64 IPython graphviz
```

In [ ]:
# Install required Python packages using pip
import sys
import importlib.util
import subprocess

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} has been installed")
    else:
        print(f"{package} is already installed")

required_packages = ['io', 'base64', 'IPython', 'graphviz']
for package in required_packages:
    install_if_missing(package)

import io
import base64
from IPython.display import display, HTML
from graphviz import Digraph, Source

io is already installed
base64 is already installed
IPython is already installed
graphviz is already installed


We'll be visualizing many directed graphs, and we'll want a function that will let us display the visualization in an accessible manner. That function is:

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Display Graph with Caption
#
# Running outside Jupyter:
#   python display_graph_with_caption.py
# ─────────────────────────────────────────────────────────────────────────────

def display_graph_with_caption(graph: Digraph, fig_num: str, title: str, alt_text: str, format: str = 'png'):
    """
    Takes a graphviz.Digraph object and displays it in a Jupyter Notebook cell
    as an accessible HTML image with a structured figure caption.

    Parameters:
    - graph (Digraph): The graphviz Digraph object to display.
    - fig_num (str): The figure number (e.g., "1", "3.1a") which is formatted as "Figure X". (UPDATED TO STRING)
    - title (str): The descriptive title for the figure caption.
    - alt_text (str): The full accessible description for screen readers.
    - format (str): The image format ('png' or 'svg' recommended). Defaults to 'png'.
    """
    if not isinstance(graph, Digraph):
        raise TypeError("The 'graph' parameter must be a graphviz.Digraph object.")

    # 1. Format Figure Number and Caption
    # fig_num is now used directly as a string
    figure_number_str = f"Figure {fig_num}"
    full_caption = f"<b>{figure_number_str}:</b> {title}"

    # 2. Get the raw binary image data using pipe()
    try:
        img_binary_data = graph.pipe(format=format)
    except Exception as e:
        print(f"Error generating graph image using pipe(): {e}")
        return

    # 3. Encode the binary data to Base64
    img_data = base64.b64encode(img_binary_data).decode('utf-8')

    # 4. Determine the mime type for the HTML
    mime_type = f"image/{format}"

    # 5. Construct the full accessible HTML block
    html_output = f"""
    <figure style="text-align: center; margin: 1em 0;">
        <img src="data:{mime_type};base64,{img_data}" alt="{alt_text}"
             style="max-width: 100%; height: auto; display: block; margin: 0 auto;" />
        <figcaption style="margin-top: 0.5em; font-size: 0.9em; color: #333;">
            {full_caption}
        </figcaption>
    </figure>
    """

    # 6. Display the HTML in the notebook
    display(HTML(html_output))

## Section 3.1 - Basic Concepts and Definitions

Let's talk about games.

In the United States two popular games for children are Candy Land and Chutes and Ladders. You may have played them when you were a kid. The basic idea for both is very simple - you move along a sequence of squares by either drawing cards or rolling dice, and the goal is to get to the end of the sequence. It's a race, but there's no strategy involved - whether you win or lose is entirely up to the luck of either the draw or the roll. It's good for little children, and it teaches them to follow directions and pay attention, but its appeal is limited because it's not possible to get "good" at playing the game.

*Note* - The games Chutes and Ladders is actually based on the ancient Indian game Snakes and Ladders that was brought to the United Kingdom in the 1890s. An image of the Snakes and Ladders game is below.

<center>
<figure>
    <img src="https://github.com/dzwick-weber/4110-book-images/blob/main/images/Snakes_and_Ladders.jpg?raw=true" height="500" alt="Old Snakes and Ladders Board">
    <figcaption><b>Figure 1.</b> <a href="https://en.wikipedia.org/wiki/Snakes_and_ladders#/media/File:Snakes_and_Ladders.jpg" target="_blank">"Snakes and Ladders"</a> is in the <a href="https://wiki.creativecommons.org/Public_domain" target="_blank">Public Domain, CC0</a></figcaption>
</figure>
</center>

What are some things these games have in common? Well, they both involve:
* A finite set of states, one of which is designated as the initial state, called the *start state*, and some of which are designated winning or *accepting states*.
* A set of possible inputs. In these cases, either the draw of a card or the roll of a dice.
* A finite set of *transitions* that tell you for each state and for each input which state to go to next.

This is, essentially, the definition of a *finite automaton* (FA for short). It's *finite* because the number of possible states, the number of possible inputs, and the number of possible transitions are all finite, and *automaton* because the change of state is totally governed by the input. In other words, it's automatic - there is no choice involved. ("Automaton" comes from Greek, and its correct plural is "automata"). The formal definition is below.

---

**Definition** - A **finite automaton** consists of:
* A finite set of states $\displaystyle Q$.
* A finite set $\displaystyle \Sigma$ of input characters, called its *alphabet*.
* A transition function $T$, where for every state $q \in Q$ and input character $a \in \Sigma$ there is a transition map $T(q,a) \rightarrow q'$ specifying the next state after the automaton in state $q$ reads the input $a$.
* A unique initial state $q_{0}$
* A subset $F \subseteq Q$ of accepting states. Note the set of accepting states could possibly be empty, possibly have more than one element, and possibly contain the initial state.

This $5$-tuple, $(Q,\Sigma,T,q_{0},F)$ defines a finite automaton.

---

## Section 3.2 - Representing Finite Automata

Let's suppose we have a much simpler game than either Candy Land or Chutes and Ladders. Let's call it the "Coin Game". Instead of rolling a dice we just flip a coin, so the possibilities at any state are just heads and tails. In other words, our input alphabet is just $\Sigma = \{H,T\}$. Our board looks like this:

<center>
<figure>
    <img src="https://github.com/dzwick-weber/4110-book-images/blob/main/images/The_Coin_Game.png?raw=true" width="600" alt="The Coin Game">
    <figcaption><b>Figure 2.</b> "The Coin Game" by Dylan Zwick (one of the authors).</figcaption>
</figure>
</center>


The idea here is that:
* The game starts at the initial state $1$ (denoted by the minus sign).
* On the first flip of the coin, the player advances to state $2$ for anything they flip.
* At state $2$, if the player flips $T$ they advance to the accepting state (denoted by the plus sign) $4$. If the player flips $H$, they advance to state $3$.
* At state $3$, if the player flips $H$ they go back to the initial state $1$. If the player flips $T$, they advance to the accepting state $4$.
* If, for some reason, the player wants to keep playing at the accepting state, they just stay there for any coin flip.

In the board diagram, the circles (or *nodes*) are the different states, and the arrows marked with letters represent how those input letters transition between states. The initial state is marked with $-$, and the accepting states are makred with $+$. This is the basic idea behind the *state diagram* of a finite automaton, which is a compact and illuminating way of representing all its features.

We could also represent the board using a *transition table*. The rows of the transition table correspond with the states of the finite automaton, the columns correspond with the possible inputs (the alphabet), and the entries represent the transitions:

<center>

| State | H | T |
| :---: | :---: | :---: |
| 1- | 2 | 2 |
| 2 | 3 | 4 |
| 3 | 1 | 4 |
| 4+ | 4 | 4 |

</center>

## Section 3.3 - Representing Finite Automata in Python

We can diagram a finite automaton in Python using the *Digraph* class from the graphviz library. The *Source* class lets us visualize it.

In [ ]:
from graphviz import Digraph, Source

We can create a class (we'll call it *coin*), and then specify the states and the transitions (called nodes and edges in the *Digraph* class). We also provide code that displays the image with a title and alternative text to help with accessibility.

In [ ]:
coin = Digraph()
coin.attr(rankdir='LR') # This specifies the diagram should go left to right

# States
coin.node('1-', shape='circle') # Initial state
coin.node('2', shape='circle')
coin.node('3', shape='circle')
coin.node('4+', shape='circle') # Accept state

# Transitions
coin.edge('1-', '2', label='H,T')
coin.edge('2', '3', label='H')
coin.edge('2', '4+', label='T')
coin.edge('3', '1-', label='H')
coin.edge('3', '4+', label='T')
coin.edge('4+', '4+', label='H,T')

In [ ]:
fig_num_3 = "3"
title_3 = "The Coin Game Finite Automaton as a Python Digraph."
alt_text_3 = (
    "A finite automaton with four states, arranged left to right. "
    "State '1-' (initial state) transitions to '2' on H or T. "
    "State '2' transitions to '3' on H, or to '4+' on T. "
    "State '3' transitions to '1-' on H, or to '4+' on T. "
    "State '4+' (accept state) self-loops on H or T. "
    "This automaton likely accepts sequences ending in T, or H then T, or starting with T."
)

display_graph_with_caption(coin, fig_num_3, title_3, alt_text_3)

Another way you'll frequently see finite automaton state diagrams is with the initial state represented by an incoming arrow that doesn't start on any node, and the  final states represented by a double circle node. Using these conventions, the state diagram above would look like:

In [ ]:
coin2 = Digraph()
coin2.attr(rankdir='LR') # This specifies the diagram should go left to right

# Start arrow
coin2.node('', shape='none')
coin2.edge('', '1')

# States
coin2.node('1', shape='circle') # Initial state
coin2.node('2', shape='circle')
coin2.node('3', shape='circle')
coin2.node('4', shape='doublecircle') # Accept state

# Transitions
coin2.edge('1', '2', label='H,T')
coin2.edge('2', '3', label='H')
coin2.edge('2', '4', label='T')
coin2.edge('3', '1', label='H')
coin2.edge('3', '4', label='T')
coin2.edge('4', '4', label='H,T')

In [ ]:
fig_num_4 = "4"
title_4 = "The Coin Game Finite Automaton as a Python Digraph with Standard Notation"
alt_text_4 = (
    "A directed graph representing a finite automaton with four states labeled 1, 2, 3, and 4. "
    "State 1 is the initial state, and State 4 is the accept state (indicated by a double circle). "
    "Transitions are as follows: "
    "From State 1, a transition goes to State 2 on input H or T. "
    "From State 2, a transition goes to State 3 on H, and a transition goes to State 4 on T. "
    "From State 3, a transition goes to State 1 on H, and a transition goes to State 4 on T. "
    "From State 4, there is a self-loop transition back to State 4 on input H or T."
)

display_graph_with_caption(coin2, fig_num_4, title_4, alt_text_4)

Now, let's create a finite automaton Python class. In order to define a finite automaton, we need to provide the tuple described in the definition from section 3.1.

In [ ]:
class FA:
    def __init__(self, states, alphabet, transition_function, start_state, accept_states):
        self.states = set(states)
        self.alphabet = set(alphabet)
        self.transition_function = transition_function
        self.start_state = start_state
        self.accept_states = set(accept_states)

        self.validate()

This initializes an FA class, with a set of states, an alphabet (set of characters), transition functions, a start state, and a set of accept states. It also runs a *validate* method to make sure it's a proper tuple and the definition makes sense. This validate method is provided below:

In [ ]:
def validate(self):
  if self.start_state not in self.states:
    raise ValueError("Start state must be in states.")
  if not self.accept_states.issubset(self.states):
    raise ValueError("Accept states must be a subset of states.")

  # Ensure total transition function
  for state in self.states:
    for symbol in self.alphabet:
      if (state, symbol) not in self.transition_function:
        raise ValueError(f"Missing transition for state '{state}' on symbol '{symbol}'.")

  # Ensure all transitions are to valid states and symbols
  for (state, symbol), next_state in self.transition_function.items():
    if state not in self.states or next_state not in self.states:
      raise ValueError(f"Invalid transition: ({state}, {symbol}) → {next_state}")
    if symbol not in self.alphabet:
      raise ValueError(f"Symbol '{symbol}' not in alphabet.")

FA.validate = validate

This method makes sure that:

* The specified initial (start) state is in the set of states.
* The specified accepting states are a subset of the set of states.
* There is a transition for every state and character in the alphabet.
* All transitions are to and from states within the set of states.

We can test this by creating the coin game FA from above:

In [ ]:
coin = FA(
    states={'1', '2', '3', '4'},
    alphabet={'H', 'T'},
    transition_function={
        ('1', 'H'): '2',
        ('1', 'T'): '2',
        ('2', 'H'): '3',
        ('2', 'T'): '4',
        ('3', 'H'): '1',
        ('3', 'T'): '4',
        ('4', 'H'): '4',
        ('4', 'T'): '4'
    },
    start_state='1',
    accept_states={'4'}
)

If we were to change this by specifying the start state to be "42", the validate method will raise an error, as "42" is not in the set of states.

In [ ]:
adams = FA(
    states={'1', '2', '3', '4'},
    alphabet={'H', 'T'},
    transition_function={
        ('1', 'H'): '2',
        ('1', 'T'): '2',
        ('2', 'H'): '3',
        ('2', 'T'): '4',
        ('3', 'H'): '1',
        ('3', 'T'): '4',
        ('4', 'H'): '4',
        ('4', 'T'): '4'
    },
    start_state='42',
    accept_states={'4'}
)

Let's also create a method that builds and returns the Digraph for the state diagram of the finite automaton. We'll use the second convention above, with the initial state represented with an incoming arrow without a source node, and the accepting states represented by double circles. We'll also merge multiple transitions from one state to another.

In [ ]:
def build_graph(self):
  dot = Digraph()
  dot.attr(rankdir='LR')
  dot.node('', shape='none')
  dot.edge('', self.start_state)

  for state in self.states:
    shape = 'doublecircle' if state in self.accept_states else 'circle'
    dot.node(state, label=str(state), shape=shape) # Ensure state labels are strings

  # Group transitions by (src, dst) to show multiple labels on one edge
  transition_groups = {}
  for (src, symbol), dst in self.transition_function.items():
    key = (src, dst)
    if key not in transition_groups:
      transition_groups[key] = []
    transition_groups[key].append(symbol)

  for (src, dst), symbols in transition_groups.items():
    label = ",".join(sorted(symbols)) # Join symbols with commas
    dot.edge(str(src), str(dst), label=label)

  return dot

FA.build_graph = build_graph

We'll also add a method for building and then displaying the state diagram.

In [ ]:
def show(self, fig_num=None, title=None, alt_text="Finite automaton diagram.", format='png'):
  """
  Generates and displays the FA digraph as an accessible HTML image.

  Parameters:
  - fig_num (str): Figure number (e.g., "Figure 3.2").
  - title (str): Title of the figure (e.g., "FA accepting binary strings").
  - alt_text (str): Accessible description for screen readers.
  - format (str): Output format for graphviz ('png', 'svg', etc.).
  """
  graph = self.build_graph()

  img_binary_data = graph.pipe(format=format)
  img_data = base64.b64encode(img_binary_data).decode('utf-8')

  caption_content = ""
  if fig_num or title:
    caption_parts = []
    if fig_num:
      caption_parts.append(f"<b>Figure {fig_num}:</b>")
    if title:
      caption_parts.append(title)

    caption_content = " ".join(caption_parts)

  #Determine the mime type for the HTML
  mime_type = f"image/{format}"


  html_output = f"""
  <figure style="text-align: center; margin: 1em 0;">
    <img src="data:{mime_type};base64,{img_data}" alt="{alt_text}"
      style="max-width: 100%; height: auto; display: block; margin: 0 auto;" />

    {f'<figcaption style="margin-top: 0.5em; font-size: 0.9em; color: #333;">{caption_content}</figcaption>' if caption_content else ''}
  </figure>
  """

  display(HTML(html_output))

FA.show = show

Let's use these methods to display the state diagram for the coin game finite automaton.

In [ ]:
fig_num_5 = "5"
title_5 = "The Coin Game from an FA Object"
alt_text_5 = (
    "This is the same image as Figure 4."
    "A directed graph representing a finite automaton with four states labeled 1, 2, 3, and 4. "
    "State 1 is the initial state, and State 4 is the accept state (indicated by a double circle). "
    "Transitions are as follows: "
    "From State 1, a transition goes to State 2 on input H or T. "
    "From State 2, a transition goes to State 3 on H, and a transition goes to State 4 on T. "
    "From State 3, a transition goes to State 1 on H, and a transition goes to State 4 on T. "
    "From State 4, there is a self-loop transition back to State 4 on input H or T."
  )

coin.show(fig_num_5, title_5, alt_text_5)

## Section 3.4 - Finite Automata and Languages

For the simple coin game above we can ask, for a given sequence of flips, whether the player wins the game - meaning whether at the end of the sequence the player is at the accepting state. For the sequence $HHHT$ the answer would be *no*. For the sequence $THT$, the answer would be *yes*. We can view the set of winning sequences of flips as a *language*, and we say the game - or finite automaton - above *accepts* this language.

For any finite automaton with input alphabet $\Sigma$, we can view a sequence of characters from $\Sigma$ as corresponding with a sequence of transitions in the finite automaton. The automaton begins at its start state, reads the first character in the string, transitions to one of its states according to the character and its transition function, and then repeats. This process continues until the finite automaton has read every character in the string. If at this point the finite automaton is in an accepting state, the string is accepted. If not, the string is rejected. We say a string that is accepted by a finite automaton is *recognized* by that finite automaton.

---

**Definition** - For a finite automaton with an input alphabet $\Sigma$, the set of strings over $\Sigma$ that are recognized by that finite automaton is the language defined by that finite automaton.

---

---

**Example** - Build a finite automaton with input alphabet $\Sigma = \{0,1\}$ that accepts any bitstring (a string of $0$s and $1$s) that ends in a $1$.

**Solution** - The language defined by the finite automaton below is the requested language.

In [ ]:
lastone = FA(
    states={'q0', 'q1'},
    alphabet={'0', '1'},
    transition_function={
        ('q0', '0'): 'q0',
        ('q0', '1'): 'q1',
        ('q1', '0'): 'q0',
        ('q1', '1'): 'q1',
    },
    start_state='q0',
    accept_states={'q1'}
)

In [ ]:
fig_num_6 = "6"
title_6 = "FA accepting any bitstring that ends in 1"
alt_text_6 = (
    "A finite automaton (FA) with two states, q0 and q1, "
    "operating over the binary alphabet {0, 1}. State q0 is the initial state, "
    "and State q1 is the single accept state (double circle). "
    "Transitions are as follows: "
    "State q0 (initial) self-loops back to q0 on input 0, and transitions to q1 on input 1. "
    "State q1 (accept) transitions to q0 on input 0, and self-loops back to q1 on input 1. "
    "This machine accepts all binary strings containing an odd number of 1s."
    )

lastone.show(fig_num_6, title_6, alt_text_6)

---

**Example** - Build an FA that accepts only those words that begin or end with a double letter.

**Solution** -

In [ ]:
doubleletter = FA(
    states={'q0', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8'},
    alphabet={'a', 'b'},
    transition_function={
        ('q0', 'a'): 'q3',
        ('q0', 'b'): 'q1',
        ('q1', 'a'): 'q5',
        ('q1', 'b'): 'q2',
        ('q2', 'a'): 'q2',
        ('q2', 'b'): 'q2',
        ('q3', 'a'): 'q4',
        ('q3', 'b'): 'q6',
        ('q4', 'a'): 'q4',
        ('q4', 'b'): 'q4',
        ('q5', 'a'): 'q7',
        ('q5', 'b'): 'q6',
        ('q6', 'a'): 'q5',
        ('q6', 'b'): 'q8',
        ('q7', 'a'): 'q7',
        ('q7', 'b'): 'q6',
        ('q8', 'a'): 'q5',
        ('q8', 'b'): 'q8',
    },
    start_state='q0',
    accept_states={'q2', 'q4', 'q7', 'q8'}
)

In [ ]:
fig_num_7 = "7"
title_7 = "FA accepting words that begin or end with a double letter"
alt_text_7 = (
    "A complex finite automaton (FA) with nine labeled states: q0, q1, q2, q3, q4, q5, q6, q7, and q8. "
    "State q0 is the initial state. The four accept states (double circle) are q2, q4, q7, and q8. "
    "The transitions are defined as follows: "
    "From q0: a leads to q3, b leads to q1. "
    "From q1: a leads to q5, b leads to q2. "
    "From q3: a leads to q4, b leads to q6. "
    "The core logic involves four accept paths: "
    "States q2 and q4 form accepting self-loops on all input symbols, meaning they are terminal accept states. "
    "States q7 and q8 are also accept states, but transitions from q7 on b and from q8 on a lead out to non-accept states."
)

doubleletter.show(fig_num_7, title_7, alt_text_7)

## Section 3.5 - Using Finite Automata In Python

In Section 3.3 we created some classes for implementing finite automata in Python and for visualizing their state diagrams. We may now want to know for a given finite automaton whether it accepts a given string over its input alphabet. Let's build a method for answering this question, and for tracing through the states encountered by the finite automaton as it processes an input string.

In [ ]:
# Method for determining whether an input string is accepted by the finite automaton.
def accepts(self, input_string):
  current_state = self.start_state
  for symbol in input_string:
    if symbol not in self.alphabet:
      raise ValueError(f"Symbol '{symbol}' not in FA alphabet.")
    current_state = self.transition_function[(current_state, symbol)]
  return current_state in self.accept_states

FA.accepts = accepts

In [ ]:
# Method for tracing through the sequence of states of the finite automaton as it processes an input string.
def trace(self, input_string):
  trace = [self.start_state]
  current_state = self.start_state
  for symbol in input_string:
    if (current_state, symbol) not in self.transition_function:
      trace.append(None)
      break
    current_state = self.transition_function[(current_state, symbol)]
    trace.append(current_state)
  return trace

FA.trace = trace

We can try these methods out on the *doubleletter* finite automaton we created above.

In [ ]:
doubleletter.accepts('abbabaab')

In [ ]:
doubleletter.accepts('abbabaabb')

In [ ]:
doubleletter.accepts('aabbabaab')

The first string neither begins nor ends with a double letter, and so is not accepted. The second string ends with a double letter, and so is accepted. The third string begins with a double letter, and so is accepted.

If we wanted to trace out the sequence of states of the *doubleletter* finite automaton as it reads the string "abbabaab" we can use the *trace* method.

In [ ]:
doubleletter.trace('abbabaab')

---

**Example** - What is the sequence of states visited by the *doubleletter* finite automaton for the input string "abbabbaab"?

**Solution** -

In [ ]:
doubleletter.trace('abbabbaab')

It's last state, 'q6', is not an accepting state, and so the string would not be accepted.



In [ ]:
doubleletter.accepts('abbabbaab')

## Section 3.6 - References and Further Reading

* Introduction to Computer Theory (Second Edition) by Daniel I.A. Cohen

  *Chapter 5 - Finite Automata*

* Automata Theory, Languages, and Computation (Third Edition) by Hopcroft, Motwani, and Ullman

  *Chapter 2 - Finite Automata*

* Introduction to the Theory of Computation (Third Edition) by Michael Sipser
  
  *Section 1.1 - Finite Automata*

* <a href="https://graphviz.org/" target="_blank">"Graphviz"</a> is licensed under the <a href="https://graphviz.org/license/" target="_blank">Common Public License Version 1.0</a>

* <a href="https://en.wikipedia.org/wiki/Snakes_and_ladders" target="_blank">"Snakes and Ladders"</a> is licensed under <a href="http://creativecommons.org/licenses/by-sa/4.0" target="_blank">CC BY-SA 4.0</a>

##  Section 3.7 - Practice Problems

These problems assume you've got a working FA class. If you've run all the code cells above, you should be good to go.

There are no separate programming problems for this chapter, as programming is integrated with the practice problems below.

### Problem 3.7.1 - Odd Parity

Let $\Sigma = \{0, 1\}$. Design a finite automaton (create an FA object in Python using the class defined above) that accepts all strings containing an odd number of 1s. (e.g., "1", "010", "111", "10101" are accepted; "0", "11", "101" are rejected). Display the state diagram for the FA.

### Problem 3.7.2 - Ending Pattern

Let $\Sigma = \{a, b\}$. Design a finite automaton (create an FA object in Python using the class defined above) that accepts any string that ends in $ba$.
(e.g., "ba", "aba", "bbba" are accepted; "b", "ab", "bab" are rejected). Display the state diagram for the FA.

### Problem 3.7.3 - The Dead State (Trap State)

Let $\Sigma = \{x, y\}$. You want to design an FA that accepts the specific string "xy" and nothing else.

* How many states do you need minimally?

* Describe the function of the "trap" (or "dead") state in this automaton. Why is it necessary according to the formal definition in Section 3.1?

### Problem 3.7.4 - Tracing an Automaton

Let $\Sigma = \{0, 1\}$. Consider the FA defined by the following transition table with start state $q_0$ and accepting state $F=\{q_1\}$.

<center>

| State | 0 | 1 |
| :---: | :---: | :---: |
| $q_{0}$ | $q_{2}$ | $q_{1}$ |
| $q_{1}$ | $q_{1}$ | $q_{0}$ |
| $q_{2}$ | $q_{0}$ | $q_{2}$ |

</center>

Implement it as an FA object in Python, and trace the following input strings. List the sequence of states visited and determine if the string is Accepted or Rejected.

* "010"

* "110"

* "00"

### Problem 3.7.5 - Divisibility

Let $\Sigma = \{0, 1\}$. However, treat the input string not as binary numbers, but just as a sequence of symbols to count.
Design an FA that accepts a string if and only if the length of the string is divisible by 3. (e.g., $\lambda$, "000", "101", "111111" are accepted). Implement this as an FA object in Python, and display the state diagram.

### Problem 3.7.6 - Formal Definition Logic

An FA is defined by the tuple $(Q, \Sigma, T, q_{0}, F)$. Suppose you have an automaton where $Q = \{A, B, C\}$ and $\Sigma = \{0, 1\}$.

* What is the exact number of entries (arrows/rows in a table) that must exist in the transition function $T$?

* If $F = \emptyset$ (the empty set), what language does this FA accept?

### Problem 3.7.7 - Contains Substring

Let $\Sigma = \{a, b\}$. Create an FA object in Python for a finite automaton that accepts any string that contains the substring "aa". Once the machine reads "aa", it should stay in an accepting state for any remaining letters. Display the state diagram for this FA.

### Problem 3.7.8 - The Empty String

Let $\Sigma = \{a, b\}$. Design a finite automaton that accepts only the empty string $\lambda$. Any non-empty string should be rejected.
(Hint: You need a start state that is also accepting, and a way to handle any input character). Implement this an an FA object in Python, and display the state diagram.